# Gold Layer: Analytics-ready datasets powering business decisions (KPIs, rankings, trends)

In [0]:
# Using Silver layer tables as input for Gold processing

customers_df = spark.table("silver_customers")
orders_df = spark.table("silver_orders")
order_items_df = spark.table("silver_order_item")
sellers_df = spark.table("silver_sellers")
reviews_df = spark.table("silver_reviews")
products_df = spark.table("silver_products")
payments_df = spark.table("silver_payments")
geolocation_df = spark.table("silver_geolocation")
full_order_df = spark.table("silver_full_order")
order_details_df = spark.table("order_details")

In [0]:
from pyspark.sql.functions import *

In [0]:
# Total value of order
order_total_value = order_details_df.groupBy("order_id")\
    .agg(sum("payment_value").alias("total_order_value"))

order_total_value.show()

# save table 
order_total_value.write.mode("overwrite").saveAsTable("order_total_value")

In [0]:
# Total Revenue Per Seller
total_revenue = full_order_df.groupBy("seller_id").agg(sum("price").alias("total_revenue")).orderBy("total_revenue",ascending=False)

total_revenue.show(5)

# save table 
total_revenue.write.mode("overwrite").saveAsTable("total_revenue")

In [0]:
# Total Orders per customer

total_orders_per_customer_df = full_order_df.groupBy("customer_id").agg(count("order_id").alias("total_orders"))
total_orders_per_customer_df.show(5)

# save table 
total_orders_per_customer_df.write.mode("overwrite").saveAsTable("total_orders_per_customer_df")

In [0]:
# Average review score per seller

review_score_df = full_order_df.groupBy("seller_id").agg(avg("review_score").alias("avg_review"))
review_score_df.show(5)

# save table 
review_score_df.write.mode("overwrite").saveAsTable("review_score")


In [0]:
# Most sold products (top 10)

top_products_df = full_order_df.groupBy("product_id")\
              .agg(count("order_item_id").alias("top_product"))\
              .orderBy("top_product",ascending=False)

top_products_df.show(10)

# save table 
top_products_df.write.mode("overwrite").saveAsTable("top_products_df")

In [0]:
# Top customers by spending

top_customer_df = full_order_df.groupBy("customer_id").agg(sum("price").alias("total_spend")).orderBy(desc("total_spend"))
top_customer_df.show(5)

# save table 
top_customer_df.write.mode("overwrite").saveAsTable("top_customer_df")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("seller_id").orderBy(desc("price"))    

# Rank top selling products per seller
top_rank_products_df = full_order_df.withColumn("rank",rank().over(window_spec)).filter(col("rank")<=5)

top_rank_products_df.show(5)

# save table 
top_rank_products_df.write.mode("overwrite").saveAsTable("top_rank_products_df")

In [0]:
# Dense rank for seller based on revenue

seller_revenue_df = full_order_df.groupBy("seller_id").agg(sum("price").alias("total_revenue"))
seller_dense_rank_window = Window.orderBy(desc("total_revenue"))
seller_rank_revenue = seller_revenue_df.withColumn("dense_rank",dense_rank().over(seller_dense_rank_window))


seller_rank_revenue.show()

# save table 
seller_rank_revenue.write.mode("overwrite").saveAsTable("seller_rank_revenue")

In [0]:
# Total revenue and Average Order Value (AOV) per customer

customer_spend_df = full_order_df.groupBy("customer_id")\
    .agg(
        count("order_id").alias("total_orders"),
        sum("price").alias("total_spent"),
        round(avg("price"),2).alias("AOV")
    )\
        .orderBy(desc("total_spent"))

customer_spend_df.show()

customer_spend_df.write.mode("overwrite").saveAsTable("revenue_AOV")

In [0]:
# Seller performance metrics (Revenue, Average review, Order count)

seller_performance_df = full_order_df.groupBy("seller_id")\
.agg(
    count("order_id").alias("total_orders"),
    sum("price").alias("total_revenue"),
    round(avg("review_score"),2).alias("avg_review"),
    round(stddev("price"),2).alias("price_variability")
)\
    .orderBy(desc("total_revenue"))

seller_performance_df.show()    


In [0]:
# Product Popularity Metrics

product_metrics_df = full_order_df.groupBy("product_id")\
    .agg(
        count("order_id").alias("total_sales"),
        sum("price").alias("total_revenue"),
        round(avg("price"),2).alias("avg_price"),
        round(stddev("price"),2).alias("price_volatility"),
        collect_set("seller_id").alias("unique_seller")
    )\
        .orderBy("total_sales")

product_metrics_df.show()     

product_metrics_df.write.mode("overwrite").saveAsTable("product_metrics")


In [0]:
# Monthly Revenue and Order Count Trend
order_month_df = full_order_df.withColumn("order_month",month("order_purchase_timestamp"))
month_order_detail_df = order_month_df.groupBy("order_month")\
    .agg(
        count("order_id").alias("total_order"),
        sum("price").alias("total_revenue"),
        round(avg("price"),2).alias("avg_order_value"),
        min("price").alias("min_revenue"),
        max("price").alias("max_revenue")

    )\
        .orderBy("order_month")

month_order_detail_df.show()

month_order_detail_df.write.mode("overwrite").saveAsTable("monthly_order_insight")        

In [0]:
# Customer Retention Analysis 

customer_retention_df = full_order_df.groupBy("customer_id")\
    .agg(
        first("order_purchase_timestamp").alias("first_order_date"),
        last("order_purchase_timestamp").alias("last_order_date"),
        count("order_id").alias("total_order"),
        sum("price").alias("total_spend"),
        round(avg("price"),2).alias("aov")

    )\
        .orderBy(desc("total_order"))

customer_retention_df.show()    


customer_retention_df.write.mode("overwrite").saveAsTable("customer_retention")

In [0]:
# Order Status Flag

order_status_flag = full_order_df\
    .withColumn("is_delivered",when(col("order_status")=="delivered",lit(1))\
        .otherwise(lit(0)))\
    .withColumn("is_canceled",when(col("order_status")=="canceled",lit(1))\
        .otherwise(lit(0)))
    

order_status_flag.display()

In [0]:
# Order Revenue Calculation

full_order_df = full_order_df.withColumn("order_revenue", col("price")+col("freight_value"))

full_order_df.select("price","freight_value","order_revenue").show()

In [0]:
# Customer segmentation based on spending

customer_spend_df = customer_spend_df.withColumn(
    "customer_segment",
    when(col("AOV")>=1200,"high_value")
    .when((col("AOV")<1200) & (col("AOV")>=600),"medium_value")
    .otherwise("low_value")
)

customer_spend_df.show()


In [0]:
# joining customer segment with full order df

full_order_df = full_order_df.join(customer_spend_df.select("customer_id","customer_segment"),"customer_id",how="left")

full_order_df.select("customer_id","customer_segment").show()


In [0]:
# Hourly Order Distribution

full_order_df = full_order_df.withColumn("hour_of_day",expr('hour(order_purchase_timestamp)'))
\
full_order_df.filter(col("customer_segment") == "medium_value") \
    .groupBy("hour_of_day") \
    .agg(sum("price").alias("total_revenue")) \
    .orderBy("hour_of_day",ascending=False) \
    .show()


In [0]:
# Weekdays vs Weekend sales

#categorise days in weekend and weekday
full_order_df = full_order_df\
    .withColumn("day_type"\
    ,when(dayofweek("order_purchase_timestamp").isin(1,7), lit("Weekend"))\
            .otherwise(lit("Weekday")))

#compare sales 

day_of_week_sales_df = full_order_df\
    .groupBy("day_type")\
    .agg(
        sum("price").alias("total_revenue"),
        count("order_id").alias("total_order"),

    ).orderBy("total_order",ascending=False)



day_of_week_sales_df.show()

day_of_week_sales_df.write.mode("overwrite").saveAsTable("weekend_vs_weekday_sales")

In [0]:
from datetime import datetime
from pyspark.sql.functions import count, sum as spark_sum, min, max

print("=" * 80)
print(f"📊 OLIST ETL PIPELINE - GOLD LAYER COMPLETION REPORT")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# Define gold tables with business context
gold_tables_info = {
    "order_total_value": "Order-level revenue: total payment value per order",
    "total_revenue": "Seller performance: cumulative revenue ranked by seller",
    "total_orders_per_customer_df": "Customer engagement: order frequency per customer",
    "review_score_df": "Seller quality: average review scores per seller",
    "top_customer_df": "VIP customers: highest-spending customers ranked",
    "top_products_df": "Best performers: top 10 products by order count",
    "top_rank_products_df": "Product quality: top-rated products",
    "product_metrics": "Product insights: sales volume, revenue, and ratings aggregated",
    "Revenue_AOV": "Customer value: average order value and revenue per customer",
    "monthly_order_insight": "Trends: monthly order volume and revenue over time",
    "customer_retention": "Loyalty analysis: repeat purchase rate and customer cohorts",
    "Hourly_Order_Distribution": "Temporal patterns: order volume by hour of day",
    "weekend_vs_weekday_sales": "Day-type analysis: sales comparison between weekdays and weekends"
}

print(f"\n✅ GOLD LAYER TABLES ({len(gold_tables_info)} total):")
print("-" * 80)
for idx, (table_name, description) in enumerate(gold_tables_info.items(), 1):
    print(f"{idx:2d}. {table_name:35s} → {description}")

# Data quality checks
print(f"\n\n📈 DATA QUALITY VERIFICATION:")
print("-" * 80)

quality_data = []
for table_name in gold_tables_info.keys():
    try:
        df = spark.read.table(table_name)
        row_count = df.count()
        col_count = len(df.columns)
        quality_data.append({
            "Table Name": table_name,
            "Rows": row_count,
            "Columns": col_count,
            "Status": "✓ OK"
        })
    except Exception as e:
        quality_data.append({
            "Table Name": table_name,
            "Rows": "N/A",
            "Columns": "N/A",
            "Status": f"✗ ERROR: {str(e)[:30]}"
        })

# Display as formatted table
quality_df = spark.createDataFrame(quality_data)
display(quality_df.orderBy("Table Name"))

print("\n" + "=" * 80)
print("✨ GOLD LAYER COMPLETE - Ready for analytics, dashboards, and reporting")
print("=" * 80)